# 🔍 Pipeline de Previsão de Fraudes
Este notebook executa uma pipeline de machine learning para prever transações suspeitas com base em dados simulados.

In [46]:
# 📦 Importações
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn import under_sampling, over_sampling
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
# 📥 Carregamento de dados
df = pd.read_csv('../data/raw/dados_coletados.csv')
print(f"Total de registros: {df.shape[0]} | Total de colunas: {df.shape[1]}")
df.head()

Total de registros: 80143 | Total de colunas: 24


,Contrato,Idade,Sexo,Valor_Renda,UF_Cliente,Perc_Juros,Prazo_Emprestimo,Data_Contratacao,Prazo_Restante,VL_Emprestimo,...,Qt_Renegociacao,Estado_Civil,Escolaridade,Possui_Patrimonio,VL_Patrimonio,QT_Parcelas_Atraso,QT_Dias_Atraso,Saldo_Devedor,Total_Pago,Possivel_Fraude
0,322090928715,42,M,4000.0,MA,17.0,140,2022-11-18,143,160000.0,...,2,SOLTEIRO(A),NaN,N,0.0,10,284.0,187861.70,1617.36,Sim
1,321990634715,31,M,3000.0,MA,20.0,28,2021-07-23,0,14000.0,...,1,CASADO (A),NaN,N,0.0,26,771.0,16615.93,1239.98,Sim
2,321965373715,36,F,2100.0,SP,24.0,180,2021-04-01,149,60000.0,...,1,CASADO (A),Nenhum,N,0.0,27,802.0,74443.40,1346.64,Sim
3,321967133715,28,M,2155.0,DF,19.0,190,2021-04-10,159,180000.0,...,2,SOLTEIRO(A),Nenhum,N,0.0,2,41.0,196812.41,22713.63,Nao
4,322098744715,21,F,4300.0,MG,22.0,100,2022-12-28,94,30000.0,...,1,SOLTEIRO(A),Ensino Médio,N,0.0,6,162.0,36114.39,900.80,Sim


In [48]:
# 🧼 Pré-processamento
df['Estado_Civil'] = df['Estado_Civil'].replace(['NENHUM'], 'OUTRO')
df['Estado_Civil'] = df['Estado_Civil'].replace(['UNIÃO ESTAVEL'], 'CASADO (A)')

bins = [0, 21, 30, 40, 50, 60, 100]
labels = ['Até 21 Anos', 'De 22 até 30 Anos', 'De 31 até 40 Anos', 'De 41 até 50 Anos', 'De 51 até 60', 'Acima de 60 Anos']
df['Faixa_Etaria'] = pd.cut(df['Idade'], bins=bins, labels=labels)

bins = [-100, 1000, 2000, 3000, 5000, 10000, 20000, 30000, 9000000000]
labels = ['Até 1k', 'De 1k até 2k', 'De 2k até 3k', 'De 3k até 5k', 'De 5k até 10k', 'De 10k até 20k',
          'De 20k até 30k', 'Acima de 50k']
df['Faixa_Salarial'] = pd.cut(df['Valor_Renda'], bins=bins, labels=labels)

bins = [-100, 30, 60, 90, 180, 240, 360, 500]
labels = ['Até 30 dias', 'De 31 até 60', 'De 61 até 90', 'De 91 até 180', 'De 181 até 240','De 241 até 360', 'Acima de 360']
df['Faixa_Dias_Atraso'] = pd.cut(df['QT_Dias_Atraso'], bins=bins, labels=labels)

bins = [0, 60, 120, 200, 720]
labels = ['Até 60 Meses', 'De 61 até 120 Meses', 'De 121 até 200 Meses', 'Acima de 200 Meses']
df['Faixa_Prazo_Emprestimo'] = pd.cut(df['Prazo_Emprestimo'], bins=bins, labels=labels)

bins = [-1, 60, 120, 200, 500]
labels = ['Até 60 Meses', 'De 61 até 120 Meses', 'De 121 até 200 Meses', 'Acima de 200 Meses']
df['Faixa_Prazo_Restante'] = pd.cut(df['Prazo_Restante'], bins=bins, labels=labels)

In [49]:
columns = ['Sexo', 'UF_Cliente', 'Perc_Juros', 
       'VL_Emprestimo', 'VL_Emprestimo_ComJuros', 'QT_Total_Parcelas_Pagas',
       'QT_Total_Parcelas_Pagas_EmDia', 'QT_Total_Parcelas_Pagas_EmAtraso',
       'Qt_Renegociacao', 'Estado_Civil', 'QT_Parcelas_Atraso', 'Saldo_Devedor', 
       'Total_Pago', 'Faixa_Prazo_Restante', 'Faixa_Salarial', 'Faixa_Prazo_Emprestimo', 'Faixa_Etaria', 
       'Faixa_Dias_Atraso', 'Possivel_Fraude']

df_dados = pd.DataFrame(df, columns=columns)

In [50]:
variaveis_categoricas = []
for i in df_dados.columns[0:18].tolist():
    if df_dados.dtypes[i] == 'object' or df_dados.dtypes[i] == 'category':                        
        variaveis_categoricas.append(i)  

In [51]:
lb = LabelEncoder()
for var in variaveis_categoricas:
    df_dados[var] = lb.fit_transform(df_dados[var])

In [52]:
PREDITORAS = df_dados.iloc[:, 0:18]  
TARGET = df_dados.iloc[:, 18] 

In [53]:
balanceador = SMOTE()
PREDITORAS_RES, TARGET_RES = balanceador.fit_resample(PREDITORAS, TARGET)

In [54]:
# 🔀 Split
X_treino, X_teste, Y_treino, Y_teste = train_test_split(PREDITORAS_RES, TARGET_RES, test_size = 0.3, random_state = 42)

In [55]:
Normalizador = MinMaxScaler()
X_treino_normalizados = Normalizador.fit_transform(X_treino)    
X_teste_normalizados = Normalizador.transform(X_teste)

In [56]:
# 🧠 Treinamento
clf = RandomForestClassifier(n_estimators  = 100, criterion = 'entropy', max_depth = 3, 
                             min_samples_leaf = 10, min_samples_split = 2)
clf = clf.fit(X_treino_normalizados, Y_treino)

In [57]:
scores = clf.score(X_treino_normalizados,Y_treino)
scores

0.9885581320709299

In [58]:
scores = clf.score(X_teste_normalizados,Y_teste)
scores

0.9860710675103905